# BCI Competition IV-2a: SSM-DAN, ablations and baselines

Chronological leave-one-subject-out on BCI IV-2a (9 subjects, 22 channels, 4 classes): in each fold the
8 other subjects are labeled sources (session T for training, session E for checkpoint selection), the
target's unlabeled session T is used in Stage 1 and its session E for IM-TTA and evaluation.

Runs on Kaggle (GPU T4 x2, **Internet on**), on Colab, or on any machine with a GPU.

1. Settings: which experiments, subjects, seed, GPUs
2. Get the code
3. Python environment (Python 3.10 with the pinned packages)
4. Download BCI IV-2a (MOABB)
5. Train: the experiments are shared over the GPUs, one experiment per GPU at a time
6. Results of the runs, subject by subject
7. Comparison table (mean +- SD, kappa, ITR, Wilcoxon + Holm)
8. Figures: per-subject accuracy and mean confusion matrices

Outputs: `results/bcic2a/<run>/seed<k>/` (Kaggle: `/kaggle/working`, Colab: `/content`), the same layout as
`python -m ssm_dan.run`. If the session ends before every fold is finished, run the notebook again with the
earlier output visible (Kaggle: *Add Input -> Your Work -> Notebook Output*; Colab: results in the mounted
Drive): finished folds are copied over and skipped.

## 1. Settings

In [ ]:
# (method, variant) pairs to run; uncomment the ones you need.
EXPERIMENTS = [
    ('ssm_dan', ''),                   # SSM-DAN; also scores the same weights without IM-TTA (Stage-1 UDA)
    # ('ssm_dan', 'no_da'),            # without Stage-1 domain adaptation (IM-TTA only)
    # ('ssm_dan', 'no_ra'),            # without Riemannian alignment
    # ('ssm_dan', 'source_only'),      # source-only backbone
    # ('ssm_dan', 'per_session_ra'),   # session-specific RA reference
    # ('eegconformer', ''),            # EEG Conformer (source-only)
    # ('atcnet', ''),                  # ATCNet (source-only)
    # ('tcformer', ''),                # TCFormer (source-only)
    # ('eegencoder', ''),              # EEGEncoder (source-only)
    # ('satransnet', ''),              # SATrans-Net (source-only)
    # ('rmetnet', ''),                 # RMETNet (domain adaptation)
    # ('ttime', ''),                   # T-TIME (test-time adaptation)
    # ('bft', ''),                     # BFT (test-time adaptation)
]
SUBJECTS = 'all'        # 'all' | '3' | '1-4' | '1,5,9'
SEED = 0                # the paper uses seed 0
GPUS = [0, 1]           # Kaggle T4 x2; one GPU: [0]. Each GPU trains one experiment at a time.
MODE = 'paper'          # 'paper': folds launched as in the reported runs (all folds of a run in one
                        # process; one process per fold for per_session_ra); 'per_fold' | 'one_process'
OVERRIDES = []          # e.g. ['max_epochs=2'] for a quick check
TIME_BUDGET_H = None    # None: 11.5 h on Kaggle/Colab, no limit elsewhere
BRANCH = 'main'

## 2. Get the code

In [ ]:
import subprocess, sys
from pathlib import Path

REPO = Path('/tmp/ssm-dan')                 # outside the saved output
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/CuongDM1806/ssm-dan.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'log', '-1', '--oneline'], check=True)
sys.path.insert(0, str(REPO))
from ssm_dan import launch, results
from ssm_dan import config as cfg

platform = launch.detect()                  # kaggle | colab | local
RESULTS = platform.results
RUNS = {cfg.resolve(method, 'bcic2a', [v for v in variant.split(',') if v], OVERRIDES)['name']: (method, variant)
        for method, variant in EXPERIMENTS}
print('platform:', platform.name, '| results:', RESULTS, '| runs:', list(RUNS))

## 3. Python environment

In [ ]:
PYTHON = launch.python_for_training(          # creates a Python 3.10 environment once if needed
    platform, ['transformers==4.48.3'] if any(m == 'eegencoder' for m, _ in EXPERIMENTS) else [])
env = launch.environment(platform)
count = launch.available_gpus()
GPUS = GPUS[:max(count or 1, 1)]
print('python:', PYTHON, '| GPUs used:', GPUS, '| MOABB data:', env['MNE_DATA'])

## 4. Download BCI IV-2a

About 1 GB of MAT files (sessions T and E of the 9 subjects) into a scratch directory, once, before training,
so that parallel runs do not download the same files at the same time.

In [ ]:
DOWNLOAD = '''
import moabb.datasets as datasets
dataset = (getattr(datasets, 'BNCI2014_001', None) or datasets.BNCI2014001)()
for subject in dataset.subject_list:
    dataset.data_path(subject)
    print('subject', subject, 'ready', flush=True)
'''
subprocess.run([str(PYTHON), '-c', DOWNLOAD], env=env, check=True)

## 5. Train

Each GPU takes the next experiment from the list and trains all its pending folds; a run's progress is
printed with its GPU and written to `results/bcic2a/<run>/seed<k>/logs/`. Before each experiment the time
left is passed on, so that no fold is started that would not finish before the session ends.

In [ ]:
import queue, threading, time

budget = TIME_BUDGET_H if TIME_BUDGET_H is not None else (11.5 if platform.name in ('kaggle', 'colab') else None)
started = time.monotonic()
todo = queue.Queue()
for run_name, experiment in RUNS.items():
    todo.put((run_name, experiment))
failed = {}

def gpu_worker(gpu):
    while True:
        try:
            run_name, (method, variant) = todo.get_nowait()
        except queue.Empty:
            return
        left = None if budget is None else budget - (time.monotonic() - started) / 3600
        if left is not None and left < 0.5:
            print(f'[GPU{gpu}] {run_name} not started: {left:.1f} h left', flush=True)
            continue
        try:
            launch.run(method, 'bcic2a', variant=variant, subjects=SUBJECTS, seed=SEED, gpus=[gpu],
                       overrides=OVERRIDES, mode=MODE, time_budget_h=left, platform=platform)
        except Exception as error:          # keep the other GPU going
            failed[run_name] = repr(error)
            print(f'[GPU{gpu}] {run_name} FAILED: {error!r}', flush=True)

threads = []
for gpu in GPUS:
    threads.append(threading.Thread(target=gpu_worker, args=(gpu,)))
    threads[-1].start()
    time.sleep(60 if len(GPUS) > 1 else 0)  # stagger the data loading of parallel runs
for thread in threads:
    thread.join()
print(f'Finished in {(time.monotonic() - started) / 3600:.2f} h.', 'Failed: ' + str(failed) if failed else '')

## 6. Results of the runs

Read from each run's `subjects/SXX.json`: accuracy (%) on session E of every target subject, the same
weights before IM-TTA (`no TTA`, the Stage-1 UDA model) when the method has IM-TTA, mean +- SD over subjects
(population SD), Cohen's kappa and training time per fold.

In [ ]:
import numpy as np

SUBJECT_IDS = list(range(1, 10))

def run_records(run_name):
    return {r['subject_id']: r for r in results.load_subjects(RESULTS / 'bcic2a' / run_name / f'seed{SEED}')}

def show(label, values):
    cells = [f'{values[s] * 100:6.2f}' if s in values else '     -' for s in SUBJECT_IDS]
    done = [values[s] * 100 for s in SUBJECT_IDS if s in values]
    mean = f'{np.mean(done):6.2f} +- {np.std(done):5.2f}' if done else ''
    print(f'{label:28s} ' + ' '.join(cells) + f'   {mean}')

print(f'{"run (accuracy %)":28s} ' + ' '.join(f'{"S" + str(s):>6s}' for s in SUBJECT_IDS) + '   mean +- SD')
for run_name in RUNS:
    records = run_records(run_name)
    if not records:
        print(f'{run_name:28s} no finished folds')
        continue
    show(run_name, {s: r['test_acc'] for s, r in records.items()})
    no_tta = {s: r['no_tta_acc'] for s, r in records.items() if 'no_tta_acc' in r}
    if no_tta:
        show('  no TTA', no_tta)
    kappa = [r['test_kappa'] for r in records.values()]
    minutes = [r.get('train_time_min', np.nan) for r in records.values()]
    print(f'{"  kappa | train time":28s} {np.mean(kappa):.3f} +- {np.std(kappa):.3f} | '
          f'{np.nanmean(minutes):.1f} min per fold | {len(records)} of {len(SUBJECT_IDS)} folds')
    print()

## 7. Comparison table

All BCI IV-2a runs under `results/` (also those of earlier sessions): per-subject accuracy, mean +- SD
(population SD over subjects, as in the paper), kappa, ITR and the exact two-sided Wilcoxon test against
`REFERENCE` with Holm's correction. The paper tests each target-data access group separately, e.g.
`REFERENCE = 'ssm_dan-source_only'` with `ONLY = 'eegconformer,atcnet,tcformer,eegencoder,satransnet'`.
Written to `results/tables/bcic2a.csv`.

In [ ]:
REFERENCE = 'ssm_dan'
ONLY = ''               # comma-separated runs; '' = every run
METRIC = 'test_acc'     # test_acc | no_tta_acc

command = [str(PYTHON), '-m', 'ssm_dan.tables', '--results', str(RESULTS), '--dataset', 'bcic2a',
           '--reference', REFERENCE, '--metric', METRIC, '--seed', str(SEED)]
subprocess.run(command + (['--runs', ONLY] if ONLY else []), cwd=REPO, env=env, check=True)

## 8. Figures

Per-subject accuracy of the runs, and each run's mean confusion matrix (every subject's matrix normalized
by the trials of each true class, then averaged over subjects, as in the paper) with its macro-F1.
Saved to `results/bcic2a/figures/`.

In [ ]:
import matplotlib.pyplot as plt

FIGURES = RESULTS / 'bcic2a' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
finished = {name: run_records(name) for name in RUNS}
finished = {name: records for name, records in finished.items() if records}

if finished:
    fig, ax = plt.subplots(figsize=(10, 3.5))
    width = 0.8 / len(finished)
    for k, (name, records) in enumerate(finished.items()):
        xs = [s + (k - (len(finished) - 1) / 2) * width for s in records]
        ax.bar(xs, [r['test_acc'] * 100 for r in records.values()], width, label=name)
    ax.axhline(25, color='gray', lw=0.8, ls='--', label='chance')
    ax.set(xticks=SUBJECT_IDS, xticklabels=[f'S{s}' for s in SUBJECT_IDS], ylabel='accuracy (%)', ylim=(0, 100),
           title='BCI IV-2a, session E')
    ax.legend(fontsize=8, ncol=min(len(finished) + 1, 4))
    fig.tight_layout()
    fig.savefig(FIGURES / f'per_subject_accuracy_seed{SEED}.png', dpi=150)
    plt.show()

    fig, axes = plt.subplots(1, len(finished), figsize=(3.6 * len(finished), 3.4), squeeze=False)
    for ax, (name, records) in zip(axes[0], finished.items()):
        classes = next(iter(records.values()))['class_names']
        counts = np.array([r['confusion_counts'] for r in records.values()], dtype=float)
        recall = counts / counts.sum(axis=2, keepdims=True)
        mean = recall.mean(axis=0) * 100
        precision = np.diagonal(counts, axis1=1, axis2=2) / counts.sum(axis=1).clip(min=1)
        diag = np.diagonal(recall, axis1=1, axis2=2)
        macro_f1 = np.mean(2 * precision * diag / (precision + diag).clip(min=1e-12), axis=1).mean()
        ax.imshow(mean, cmap='Blues', vmin=0, vmax=100)
        for i in range(len(classes)):
            for j in range(len(classes)):
                ax.text(j, i, f'{mean[i, j]:.1f}', ha='center', va='center', fontsize=8,
                        color='white' if mean[i, j] > 50 else 'black')
        short = [c.replace('_', ' ') for c in classes]
        ax.set(xticks=range(len(classes)), yticks=range(len(classes)), xticklabels=short, yticklabels=short,
               xlabel='predicted', ylabel='true', title=f'{name}\nmacro-F1 {macro_f1 * 100:.2f}')
        ax.tick_params(axis='x', labelrotation=45, labelsize=8)
        ax.tick_params(axis='y', labelsize=8)
    fig.tight_layout()
    fig.savefig(FIGURES / f'mean_confusion_seed{SEED}.png', dpi=150)
    plt.show()
else:
    print('No finished folds yet.')

Per-fold confusion matrices and learning curves are in each run's `figures/`, the trained checkpoints in
`checkpoints/`, the resolved configuration in `config.yaml`. On Kaggle, save a version of the notebook to keep
`/kaggle/working`; attach that output to the next session to continue unfinished runs.